### ==============================================================================
## Processing of Moving Vessel Profiler Data - code 5.1 (Data Generation)
### Authors: Elisabet Verger-Miralles (everger@imedea.uib-csic.es)
### Data from BioSWOT experiment 
# 
**DESCRIPTION**:
This script finalizes the processed datasets for publication. It applies 
a final static salinity offset (determined via CTD calibration), calculates 
standard TEOS-10 variables (Absolute Salinity, Conservative Temperature, 
Potential Density) using the GSW library, recalculates a consistent 
final conductivity, and structures the NetCDF file with official standard 
names and CF-compliant metadata.
#
INPUT: Smoothed NetCDF files (*_step4_loess.nc).
#
OUTPUT: Final NetCDF datasets (*_step5_final.nc).
### ==============================================================================

In [ ]:


import xarray as xr
import numpy as np
import gsw
from pathlib import Path

# ==========================================
# 1. CONFIGURATION BIOSWOT
# ==========================================
BASE_ROOT = Path(r"C:\Users\ASUS\Desktop\MVP\MVP_paper_data_figures_publish\MVP_bioswot_nc_final_processing")
DIR_IN = BASE_ROOT / "Data" / "processed_step4_loess"
DIR_OUT = BASE_ROOT / "Data" / "processed_step5_final"
DIR_OUT.mkdir(parents=True, exist_ok=True)

# Offset = 0 for BioSWOT
FINAL_OFFSET = 0.0 

print(f"Starting BioSWOT Step 5 (Generating publishable files)...")

# ==========================================
# 2. PROCESSING
# ==========================================
files = sorted(list(DIR_IN.glob("*_step4_loess.nc")))

if not files:
    print(f"❌ No files found in {DIR_IN}. Check Step 4.")
else:
    for f in files:
        try:
            with xr.open_dataset(f) as ds:
                p = ds['pressure'].values
                t_smooth = ds['t1_smooth'].values
                s_smooth = ds['salinity_smooth'].values
                lat, lon = ds.latitude.values, ds.longitude.values

                # 1. Salinity  Final (Offset = 0)
                s_final = s_smooth + FINAL_OFFSET
                
                # 2. Variables TEOS-10
                SA = gsw.SA_from_SP(s_final, p, lon, lat)
                CT = gsw.CT_from_t(SA, t_smooth, p)
                sigma0 = gsw.sigma0(SA, CT)
                
                # 3. Conductivity consistent with the smoothed salinity
                c_final = gsw.C_from_SP(s_final, t_smooth, p)

                # 4. Final Dataset
                ds_final = ds.copy(deep=True)
                
                ds_final['temperature_conservative'] = (('scan',), CT)
                ds_final['salinity_practical'] = (('scan',), s_final)
                ds_final['conductivity_final'] = (('scan',), c_final)
                ds_final['sigma_theta'] = (('scan',), sigma0)
                
                # Metadata
                ds_final['temperature_conservative'].attrs = {
                    'units': 'degC', 
                    'standard_name': 'sea_water_conservative_temperature',
                    'long_name': 'Conservative Temperature'
                }
                ds_final['salinity_practical'].attrs = {
                    'units': 'PSU', 
                    'standard_name': 'sea_water_practical_salinity', 
                    'comment': f'No offset applied for BioSWOT ({FINAL_OFFSET})'
                }
                ds_final['conductivity_final'].attrs = {
                    'units': 'mS/cm', 
                    'standard_name': 'sea_water_electrical_conductivity'
                }
                ds_final['sigma_theta'].attrs = {
                    'units': 'kg/m^3', 
                    'standard_name': 'sea_water_sigma_theta',
                    'long_name': 'Potential Density Anomaly (sigma-0)'
                }

                out_name = f.name.replace('_step4_loess.nc', '_step5_final.nc')
                ds_final.to_netcdf(DIR_OUT / out_name)
                
            print(f"   ✅ {f.name} -> {out_name}")
            
        except Exception as e:
            print(f"   ❌ Error in {f.name}: {e}")

print(f"\n DONE! {len(files)} files BioSWOT ready {DIR_OUT}")

c:\Users\ASUS\anaconda3\envs\env_elisabet\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


🚀 Iniciando Paso 5 BioSWOT (Generando archivos publicables)...
   ✅ PL10_mvp_2023-04-29_224406_step4_loess.nc -> PL10_mvp_2023-04-29_224406_step5_final.nc
   ✅ PL10_mvp_2023-04-29_225254_step4_loess.nc -> PL10_mvp_2023-04-29_225254_step5_final.nc
   ✅ PL10_mvp_2023-04-29_230112_step4_loess.nc -> PL10_mvp_2023-04-29_230112_step5_final.nc
   ✅ PL10_mvp_2023-04-29_230930_step4_loess.nc -> PL10_mvp_2023-04-29_230930_step5_final.nc
   ✅ PL10_mvp_2023-04-29_231748_step4_loess.nc -> PL10_mvp_2023-04-29_231748_step5_final.nc
   ✅ PL10_mvp_2023-04-29_232609_step4_loess.nc -> PL10_mvp_2023-04-29_232609_step5_final.nc
   ✅ PL10_mvp_2023-04-29_233428_step4_loess.nc -> PL10_mvp_2023-04-29_233428_step5_final.nc
   ✅ PL10_mvp_2023-04-29_234249_step4_loess.nc -> PL10_mvp_2023-04-29_234249_step5_final.nc
   ✅ PL10_mvp_2023-04-29_235113_step4_loess.nc -> PL10_mvp_2023-04-29_235113_step5_final.nc
   ✅ PL10_mvp_2023-04-29_235937_step4_loess.nc -> PL10_mvp_2023-04-29_235937_step5_final.nc
   ✅ PL10_mvp_202

### ==============================================================================
## Processing of Moving Vessel Profiler Data - code 5.2 (Validation Figures)
### Authors: Elisabet Verger-Miralles (everger@imedea.uib-csic.es) 
### Data from BioSWOT experiment 
# 
**DESCRIPTION**:
This script generates the final publication-ready figures comparing the 
fully calibrated MVP profiles against their nearest CTD casts. The 3-panel 
plots display Temperature, Conductivity, and Salinity, illustrating the raw 
MVP data, the final processed MVP data, and the CTD. A text box reports the 
percentage of error reduction achieved by the pipeline.
#
INPUT: Raw MVP files (Step 1), Final MVP files (Step 5), and Reference CTD (.cnv).
#
OUTPUT: Final comparison plots (.png) in 'FIGURES_PAPER_FINAL_RESULTS'.
### ==============================================================================

In [ ]:

import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
import pandas as pd
from pathlib import Path
from geopy.distance import geodesic
import gsw
import re
import io
import warnings

warnings.filterwarnings("ignore")

# ==========================================
# 1. CONFIGURATION BIOSWOT
# ==========================================
BASE_ROOT = Path(r"C:\Users\ASUS\Desktop\MVP\MVP_paper_data_figures_publish\MVP_bioswot_nc_final_processing")

DIR_MVP_RAW = BASE_ROOT / "Data" / "processed_step1_highres_qc"
DIR_MVP_FINAL = BASE_ROOT / "Data" / "processed_step5_final" 
DIR_CTD = Path(r"C:\Users\ASUS\Desktop\BioSWOT_data\CTD\cnv")

OUTDIR = BASE_ROOT / "Figures" / "FIGURES_PAPER_FINAL_BIOSWOT"
OUTDIR.mkdir(parents=True, exist_ok=True)

# Strict parameters for final validation
MAX_DIST_KM = 2.5    
MAX_TIME_MIN = 120.0  
Z_MAX_PLOT = 350.0   # BioSWOT reaches greater depth
CTD_SMOOTH_WINDOW = 8 

# ==========================================
# 2. SUPPORT FUNCTIONS 
# ==========================================

def read_ctd_robust(path):
    try:
        with open(path, 'r', encoding='latin-1') as f: lines = f.readlines()
        lat, lon, time_val, start_idx = np.nan, np.nan, pd.NaT, 0
        col_map = {}
        for i, line in enumerate(lines):
            if 'NMEA Latitude' in line:
                p = re.findall(r"[-+]?\d*\.\d+|\d+", line)
                if len(p)>=2: lat = float(p[0]) + float(p[1])/60 * (-1 if 'S' in line else 1)
            if 'NMEA Longitude' in line:
                p = re.findall(r"[-+]?\d*\.\d+|\d+", line)
                if len(p)>=2: lon = float(p[0]) + float(p[1])/60 * (-1 if 'W' in line else 1)
            if 'start_time' in line:
                try: time_val = pd.to_datetime(line.split('=')[1].strip().split('[')[0])
                except: pass
            if '# name' in line:
                match = re.search(r"name (\d+)", line)
                if match:
                    idx = int(match.group(1)); name = line.split('=')[1].split(':')[0].strip().lower()
                    if any(x in name for x in ['prdm', 'pres']): col_map['p'] = idx
                    elif any(x in name for x in ['t090', 't190', 'temp']): col_map['t'] = idx
                    elif any(x in name for x in ['sal00', 'sal11', 'sal']): col_map['s'] = idx
                    elif any(x in name for x in ['c0', 'c1', 'cond']): col_map['c'] = idx
            if '*END*' in line: start_idx = i+1; break
        
        df = pd.read_csv(io.StringIO("".join(lines[start_idx:])), delim_whitespace=True, header=None)
        p, t, s = df.iloc[:, col_map['p']].values, df.iloc[:, col_map['t']].values, df.iloc[:, col_map['s']].values
        c = df.iloc[:, col_map['c']].values if 'c' in col_map else None
        if c is not None and np.nanmean(c) < 10: c *= 10.0
        return {'lat': lat, 'lon': lon, 'time': time_val, 'id': path.stem, 'p': p, 't': t, 's': s, 'c': c}
    except: return None

def load_mvp_catalog(mvp_dir):
    data = []
    files = sorted(list(mvp_dir.glob("*.nc")))
    for f in files:
        try:
            with xr.open_dataset(f) as ds:
                lat = np.nanmean(ds['lat'].values if 'lat' in ds else ds.latitude.values)
                lon = np.nanmean(ds['lon'].values if 'lon' in ds else ds.longitude.values)
                t_val = pd.to_datetime(ds.time.values.flatten()[0]) if 'time' in ds else pd.NaT
                data.append({'file': f.name, 'lat': lat, 'lon': lon, 'time': t_val})
        except: pass
    return pd.DataFrame(data)

# ==========================================
# 3. PLOTTING
# ==========================================

df_mvp = load_mvp_catalog(DIR_MVP_RAW)

for ctd_p in sorted(DIR_CTD.glob("*.cnv")):
    ctd = read_ctd_robust(ctd_p)
    if not ctd or pd.isna(ctd['time']): continue
    
    # Match 
    df_mvp['dist_km'] = [geodesic((ctd['lat'], ctd['lon']), (m.lat, m.lon)).km for m in df_mvp.itertuples()]
    df_mvp['dt_min'] = [(abs(m.time - ctd['time']).total_seconds() / 60.0) for m in df_mvp.itertuples()]
    matches = df_mvp[(df_mvp['dist_km'] <= MAX_DIST_KM) & (df_mvp['dt_min'] <= MAX_TIME_MIN)]
    
    if matches.empty: continue
            
    best_match = matches.loc[matches['dist_km'].idxmin()]
    raw_path = DIR_MVP_RAW / best_match['file']
    final_path = DIR_MVP_FINAL / best_match['file'].replace("_step1_qc.nc", "_step5_final.nc")

    if not final_path.exists(): continue

    try:
        with xr.open_dataset(final_path) as ds_f, xr.open_dataset(raw_path) as ds_r:
            # MVP Processing (Step 5)
            p_p, t_p, s_p, c_p = ds_f.pressure.values, ds_f.temperature_conservative.values, ds_f.salinity_practical.values, ds_f.conductivity_final.values
            
            # MVP Raw (Step 1)
            p_r, t_r, c_r = ds_r.pressure.values, ds_r.t1.values, ds_r.c1.values
            if np.nanmean(c_r) < 10: c_r *= 10
            s_raw = gsw.SP_from_C(c_r, t_r, p_r)

            # --- METRICS OF ERROR REDUCTION ---
            p_grid = np.arange(10, Z_MAX_PLOT, 1.0)
            s_ctd_smooth = pd.Series(ctd['s']).rolling(window=CTD_SMOOTH_WINDOW, center=True, min_periods=1).mean().values
            s_ctd_i = np.interp(p_grid, ctd['p'], s_ctd_smooth)
            s_raw_i = np.interp(p_grid, p_r, s_raw)
            s_fin_i = np.interp(p_grid, p_p, s_p)
            
            mask = np.isfinite(s_ctd_i) & np.isfinite(s_raw_i) & np.isfinite(s_fin_i)
            rmsd_raw = np.sqrt(np.mean((s_ctd_i[mask] - s_raw_i[mask])**2))
            rmsd_fin = np.sqrt(np.mean((s_ctd_i[mask] - s_fin_i[mask])**2))
            error_red = (1 - rmsd_fin/rmsd_raw) * 100

            # --- PLOT ---
            fig, axs = plt.subplots(1, 3, figsize=(16, 7), sharey=True, constrained_layout=True)
            kw_raw = {'s': 5, 'color': '#CCCCCC', 'alpha': 0.7, 'label': 'MVP Raw'}
            kw_fin = {'s': 5, 'color': 'red', 'alpha': 0.8, 'label': f'MVP Processed'}

            # (A) Temperature
            axs[0].scatter(t_r, p_r, **kw_raw)
            axs[0].scatter(t_p, p_p, **kw_fin)
            axs[0].plot(ctd['t'], ctd['p'], color='black', lw=2, label='CTD Reference')
            axs[0].set_xlabel("Temperature (°C)")
            axs[0].set_ylabel("Pressure (dbar)")

            # (B) Conductivity
            axs[1].scatter(c_r, p_r, **kw_raw)
            axs[1].scatter(c_p, p_p, **kw_fin)
            if ctd['c'] is not None:
                axs[1].plot(ctd['c'], ctd['p'], color='black', lw=2)
            axs[1].set_xlabel("Conductivity (mS/cm)")

            # (C) Salinity
            axs[2].scatter(s_raw, p_r, **kw_raw)
            axs[2].scatter(s_p, p_p, **kw_fin)
            axs[2].plot(s_ctd_smooth, ctd['p'], color='black', lw=2.5, label='CTD Reference')
            axs[2].set_xlim(37.7,38.7)
            axs[2].set_xlabel("Practical Salinity (PSU)")
            axs[2].legend(loc='lower right', fontsize=9)

            for ax in axs: ax.invert_yaxis(); ax.set_ylim(Z_MAX_PLOT, 5); ax.grid(True, alpha=0.3)

            txt = (f"CTD: {ctd['id']}\nDist: {best_match['dist_km']:.2f} km\nDt: {best_match['dt_min']:.1f} min\nRMSD Red: {error_red:.1f}%")
            axs[0].text(0.05, 0.05, txt, transform=axs[0].transAxes, fontsize=10, bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))

            plt.savefig(OUTDIR / f"BioSWOT_Validation_{ctd['id']}.png", dpi=300)
            plt.close()
            print(f"   ✅ Validation BioSWOT {ctd['id']} OK.")

    except Exception as e:
        print(f"   ❌ Error in {ctd['id']}: {e}")

print(f"\n Check the unified figures in {OUTDIR}")

🚀 Iniciando validación BioSWOT...
   ✅ Validación BioSWOT bioswot2023_0017 OK.
   ✅ Validación BioSWOT bioswot2023_0018 OK.
   ✅ Validación BioSWOT bioswot2023_0019 OK.
   ✅ Validación BioSWOT bioswot2023_0024 OK.
   ✅ Validación BioSWOT bioswot2023_0038 OK.

✨ ¡Finalizado! Revisa las figuras unificadas en C:\Users\ASUS\Desktop\MVP\MVP_paper_data_figures_publish\MVP_bioswot_nc_final_processing\FIGURES_PAPER_FINAL_BIOSWOT
